# 📅 Exercício Extra 4: Estacionamento por AprilTags / Marcadores 🅿️🤖

Nas fábricas inteligentes (como os armazéns da Amazon), os robôs guiam-se por símbolos colados no chão chamados **AprilTags**. São parecidos com códigos QR, mas muito mais fáceis e rápidos de ler pelas câmaras dos robôs.

Cada tag tem um número de ID único. Vamos programar o robô para encontrar o local de estacionamento marcado com a **Tag ID = 0**.

**Nota Prévia:** Esta atividade requer a biblioteca de deteção instalada. (Se não estiver instalada, corre primeiro numa célula: `!pip3 install pupil-apriltags`)

### 🎯 O Teu Objetivo
Mostrar a AprilTag impressa à câmara do JetRacer e ler no ecrã a distância e o centro exato para onde o carro teria de navegar.

### 🛠️ Instruções
1. Imprime ou mostra no ecrã do telemóvel uma imagem de uma "AprilTag da família 36h11" (ID 0).
2. Corre o código abaixo e aponta a câmara para a tag.

In [ ]:
import cv2
import ipywidgets as widgets
from IPython.display import display
from jetcam.csi_camera import CSICamera
import time

# Tenta importar o detetor industrial da AprilTag
try:
    from pupil_apriltags import Detector
    detetor = Detector(families='tag36h11')
    print("--- SISTEMA DE ESTACIONAMENTO ASSISTIDO ATIVO ---")
except ImportError:
    print("⚠️ Instala primeiro a biblioteca correndo: !pip3 install pupil-apriltags")

camera = CSICamera(width=300, height=300, capture_width=1280, capture_height=720, capture_fps=15)
imagem_widget = widgets.Image(format='jpeg', width=300, height=300)
botao_desligar = widgets.Button(description="❌ DESLIGAR", button_style='danger')

display(imagem_widget, botao_desligar)
sistema_ativo = True

def detetar_tags(change):
    global sistema_ativo
    if not sistema_ativo: return
    
    frame = change['new']
    cinzento = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    
    # Procura marcadores na imagem cinzenta
    tags = detetor.detect(cinzento)
    
    for tag in tags:
        # Extrai os cantos do marcador quadrado detetado
        (ptA, ptB, ptC, ptD) = tag.corners
        ptB = (int(ptB[0]), int(ptB[1]))
        ptD = (int(ptD[0]), int(ptD[1]))
        
        # Desenha um retângulo verde à volta da tag
        cv2.rectangle(frame, ptB, ptD, (0, 255, 0), 2)
        
        # Identifica qual é o ID da vaga de estacionamento
        tag_id = tag.tag_id
        centro_x = int(tag.center[0])
        
        # Escreve a informação no ecrã
        texto = f"Lugar ID: {tag_id} | Centro X: {centro_x}"
        cv2.putText(frame, texto, (int(ptB[0]), int(ptB[1]) - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
        
        if tag_id == 0:
            print(f"🅿️ Vaga Correta Detetada! Alinhando ao Centro X: {centro_x}   ", end='\r')
            
    _, jpeg = cv2.imencode('.jpg', frame)
    imagem_widget.value = jpeg.tobytes()
    time.sleep(0.02)

camera.observe(detetar_tags, names='value')

def encerra(b):
    global sistema_eclipse
    sistema_ativo = False
    camera.unobserve(detetar_tags, names='value')
    camera.running = False
    print("Sistema desligado.")

botao_desligar.on_click(encerra)
camera.running = True